# 6. Uncertainty: conformal sets and evidential rule trees

A point prediction hides how sure the model is. Ex-Fuzzy offers two ways to
predict *sets* of classes instead:

- **Conformal prediction** wraps any fuzzy classifier and calibrates it on
  held-out data so that the set contains the true class with probability at
  least 1 - alpha, whatever the model.
- **FERL** and **DeepFERL** learn fuzzy rule trees whose outputs are
  Dempster-Shafer evidence: belief, plausibility and ignorance per class,
  and prediction sets without any calibration step.

The data is the Wisconsin breast cancer set from scikit-learn.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, DeepFERL, FERL, fuzzy_sets as fs, utils
from ex_fuzzy.conformal import ConformalFuzzyClassifier, evaluate_conformal_coverage

cancer = load_breast_cancer(as_frame=True)
X = cancer.frame.drop(columns='target')
y = cancer.target_names[cancer.target]                       # 'malignant' / 'benign'

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0, stratify=y_rest)
print(f'train {len(X_train)}, calibration {len(X_cal)}, test {len(X_test)} samples; {X.shape[1]} features')

train 341, calibration 114, test 114 samples; 30 features


## Conformal prediction

The wrapper fits the classifier on the training split and calibrates on the
calibration split. Never calibrate on training data: the guarantee relies
on the calibration samples being exchangeable with the test samples.

In [2]:
base = BaseFuzzyRulesClassifier(nRules=10, nAnts=3, linguistic_variables=utils.construct_partitions(X_train, fs.FUZZY_SETS.t1),
                                n_gen=30, pop_size=30, random_state=0)
conformal = ConformalFuzzyClassifier(base, score_type='membership')
conformal.fit(X_train, y_train, X_cal, y_cal)
print(f'point-prediction accuracy: {conformal.score(X_test, y_test):.3f}')

rows = []
for alpha in (0.05, 0.1, 0.2):
    metrics = evaluate_conformal_coverage(conformal, X_test, y_test, alpha=alpha)
    rows.append({'alpha': alpha, 'target coverage': 1 - alpha, 'coverage': round(metrics['coverage'], 3),
                 'average set size': round(metrics['avg_set_size'], 2), 'singletons': round(metrics['singleton_sets'], 2),
                 'empty sets': round(metrics['empty_sets'], 2)})
pd.DataFrame(rows)

point-prediction accuracy: 0.921


,alpha,target coverage,coverage,average set size,singletons,empty sets
0,0.05,0.95,1.000,2.00,0.00,0.00
1,0.10,0.90,0.877,1.70,0.30,0.00
2,0.20,0.80,0.772,0.94,0.68,0.19


Smaller alpha buys more coverage with larger sets. A singleton is a
confident answer; a two-class set says "not sure"; an empty set says the
sample looks unlike anything calibrated.

In [3]:
sets = conformal.predict_set(X_test.head(8), alpha=0.1)
pd.DataFrame({'true': y_test[:8], 'point prediction': conformal.predict(X_test.head(8)), 'prediction set': [sorted(s) for s in sets]})

,true,point prediction,prediction set
0,benign,benign,"[benign, malignant]"
1,benign,benign,"[benign, malignant]"
2,malignant,malignant,[malignant]
3,malignant,malignant,"[benign, malignant]"
4,malignant,malignant,"[benign, malignant]"
5,malignant,malignant,"[benign, malignant]"
6,benign,benign,"[benign, malignant]"
7,benign,benign,[malignant]


Each set comes with the rules behind it and the p-value of every class.

In [4]:
explanation = conformal.predict_set_with_rules(X_test.head(1), alpha=0.1)[0]
print('p-values:', {label: round(value, 3) for label, value in explanation['class_p_values'].items()})
print('prediction set:', explanation['prediction_set'])
for contribution in explanation['rule_contributions'][:3]:
    print(f"  rule {contribution['rule_index']} -> {contribution['class']}: firing {contribution['firing_strength']:.3f}, "
          f"confidence {contribution['rule_confidence']:.2f}")

p-values: {'benign': np.float64(1.0), 'malignant': np.float64(0.136)}
prediction set: {'benign', 'malignant'}
  rule 1 -> benign: firing 0.260, confidence 1.00
  rule 2 -> benign: firing 0.143, confidence 1.00
  rule 3 -> benign: firing 0.000, confidence 0.77


The three nonconformity scores (`'membership'`, `'association'`,
`'entropy'`) can be compared on the same fitted model without refitting,
since only the calibration changes.

In [5]:
rows = []
for score_type in ('membership', 'association', 'entropy'):
    variant = ConformalFuzzyClassifier(conformal.clf, score_type=score_type).calibrate(X_cal, y_cal)
    metrics = evaluate_conformal_coverage(variant, X_test, y_test, alpha=0.1)
    rows.append({'score': score_type, 'coverage': round(metrics['coverage'], 3), 'average set size': round(metrics['avg_set_size'], 2)})
pd.DataFrame(rows)

,score,coverage,average set size
0,membership,0.877,1.70
1,association,0.877,1.70
2,entropy,0.921,1.74


## FERL: evidence from a fuzzy rule tree

FERL grows a tree of fuzzy splits greedily and treats each leaf as a source
of evidence. `predict_credal` returns the pignistic probability, the belief
and plausibility of every class, and the ignorance mass, which is how much
evidence points nowhere in particular.

In [6]:
ferl = FERL(max_rules=15, random_state=0)
ferl.fit(X_train, y_train)
print(f'FERL accuracy: {ferl.score(X_test, y_test):.3f}, {ferl.n_rules()} rules')

betp, belief, plausibility, ignorance = ferl.predict_credal(X_test.head(6))
pd.DataFrame({'true': y_test[:6], 'prediction': ferl.predict(X_test.head(6)),
              **{f'belief {label}': belief[:, index].round(2) for index, label in enumerate(ferl.classes_)},
              **{f'plausibility {label}': plausibility[:, index].round(2) for index, label in enumerate(ferl.classes_)},
              'ignorance': np.round(ignorance, 2)})

FERL accuracy: 0.904, 2 rules


,true,prediction,belief benign,belief malignant,plausibility benign,plausibility malignant,ignorance
0,benign,benign,0.90,0.10,0.90,0.10,0.00
1,benign,benign,0.80,0.09,0.91,0.20,0.12
2,malignant,malignant,0.04,0.96,0.04,0.96,0.00
3,malignant,malignant,0.03,0.83,0.17,0.97,0.14
4,malignant,malignant,0.04,0.96,0.04,0.96,0.00
5,malignant,malignant,0.04,0.96,0.04,0.96,0.00


In [7]:
ferl.print_tree()

FERL tree (max_rules=15, coverage_threshold=0.0)
└── Root: class=benign, coverage=1.000
    ├── root_F22_L2: worst perimeter IS High → class=malignant, coverage=0.285, split_criterion=0.467
    │   └── root_F22_L2_F6_L0: mean concavity IS Low → class=malignant, coverage=0.003, split_criterion=0.006
    └── root_F4_L0: mean smoothness IS Low → class=benign, coverage=0.294, split_criterion=0.025
        └── root_F4_L0_F4_L1: mean smoothness IS Medium → class=benign, coverage=0.034, split_criterion=0.009


## DeepFERL, and the three set predictors side by side

`DeepFERL` grows a deeper tree of learned soft splits and votes over its
leaves. Its sets are evidential like FERL's. The empirical coverage below is
the fraction of test samples whose true class is in the set; only the
conformal sets carry a guarantee.

In [8]:
deep = DeepFERL(random_state=0)
deep.fit(X_train, y_train)


def set_summary(name, model, class_sets):
    true_index = np.searchsorted(model.classes_, y_test)
    return {'model': name, 'accuracy': round(model.score(X_test, y_test), 3),
            'coverage': round(float(class_sets[np.arange(len(y_test)), true_index].mean()), 3),
            'average set size': round(float(class_sets.sum(axis=1).mean()), 2)}


conformal_sets = conformal.predict_set(X_test, alpha=0.1)
conformal_matrix = np.array([[label in s for label in conformal.clf.classes_] for s in conformal_sets])
pd.DataFrame([
    set_summary('conformal (alpha=0.1)', conformal.clf, conformal_matrix),
    set_summary('FERL', ferl, ferl.predict_set(X_test)),
    set_summary('DeepFERL', deep, deep.predict_set(X_test)),
])

,model,accuracy,coverage,average set size
0,conformal (alpha=0.1),0.921,0.877,1.70
1,FERL,0.904,0.974,1.52
2,DeepFERL,0.974,0.974,1.03
